# Bash 5.3 deep-dive: GLOBSORT, inherit_errexit, ${| command;}, and test builtin changes

This notebook isolates four Bash 5.3 behavior changes through runnable probes. Each section shows a minimal reproduction, explains the behavior, and includes a version guard so 5.3-only probes self-skip on older runtimes.

> **last_verified:** 2026-10-08  
> **tool_version:** Bash 5.3 (patches 1-20; master HEAD = patch 20, committed 2026-09-15)  
> **sources:** https://www.phoronix.com/news/GNU-Bash-5.3, https://cgit.git.savannah.gnu.org/cgit/bash.git/log/?h=master, https://tiswww.case.edu/php/chet/bash/CHANGES, https://lists.gnu.org/archive/html/bash-announce/2025-07/msg00000.html, https://lists.gnu.org/archive/html/bash-announce/2025-07/msg00005.html, https://cgit.git.savannah.gnu.org/cgit/bash.git/plain/NEWS

Patch window note: upstream bash-5.3 is at patchlevel 20 (committed 2026-09-15); patches 16-20 are readline/TTY fixes. The four behavior changes below are language-level, present from the initial 5.3 release, and stable across the patch window.


## Purpose

Four Bash 5.3 behavior changes affect everyday scripting but are easy to miss until a test fails: pathname-expansion sort order (`GLOBSORT`), error-propagation scope (`inherit_errexit`), a new forkless command-substitution form (`${| command;}`), and how `test`/`[` parses complex parenthesized expressions. This notebook isolates each change with a minimal probe so the behavior can be verified before relying on it in production scripts.


## When to use

| Situation | Action |
|---|---|
| Writing a deploy script that globs versioned artifacts | Enable `shopt -s GLOBSORT` behind a version guard and verify ordering. |
| Tightening a `set -e` script's failure detection | Enable `shopt -s inherit_errexit` and audit command substitutions that previously swallowed exit codes. |
| Eliminating a pipe for a simple capture | Use `${ command; }` instead of `$(command)` to avoid a child process. |
| Porting a complex `[ ]` expression to 5.2->5.3 | Re-test expressions with parenthesized subexpressions and >4 args; parsing changed. |
| CI on a mixed-version fleet | Run every code cell on both runtimes and diff outputs -- see Verify. |


## Prerequisites

- **Bash 5.3** for the 5.3-only probes; earlier versions self-skip via the `BASH_VERSINFO` guard in cell 0.
- **`bash -n`** for syntax validation of each code cell.
- **ShellCheck** (`0.10.0` or later) for static linting of the snippets.
- The version-guard idiom used throughout:
  ```bash
  if (( BASH_VERSINFO[0] >= 5 && BASH_VERSINFO[1] >= 3 )); then
    # 5.3-only probe here
  else
    echo "SKIP: needs Bash 5.3+"
  fi
  ```


## Key terminology

- **GLOBSORT** -- A `shopt` option (Bash 5.3+) that changes pathname-expansion sort order; the option name is unrecognized on Bash < 5.3. Run the code cell below to observe the difference.
- **inherit_errexit** -- A `shopt` option that makes `set -e` propagate into subshells and command substitutions, closing the gap where `$(cmd)` appears to succeed. Guard with a `BASH_VERSINFO` check for cross-version safety.
- **Forkless command substitution** -- `${ command; }` and `${|command;}` capture a command's output in the current shell context without forking a child process or using a pipe. Introduced in Bash 5.3.
- **pipefail relationship** -- Because `${| command;}` avoids pipes entirely, it sidesteps `PIPESTATUS`/`pipefail` exit-status ambiguity; the inner command's exit code is directly available as `$?` after the substitution.
- **Parenthesized subexpression** -- A `(` ... `)` group inside a `test`/`[ ]` argument list that creates an implicit subexpression; 5.3 changed how >4-arg forms are parsed.
- **posixtest** -- The Bash `test` internal function invoked for POSIX-compatible evaluation of parenthesized subexpressions; 5.3 routes the subexpression to it when four or fewer args sit between the parens.


## Version guard pattern

All 5.3-only probes below are guarded by `BASH_VERSINFO`. On a 5.2 runtime the cells print `SKIP` instead of failing mid-script.


In [ ]:
# Cell 0 -- version guard
echo "bash: $(bash --version | head -1)"
echo "BASH_VERSINFO: ${BASH_VERSINFO[0]}.${BASH_VERSINFO[1]} patchlevel ${BASH_PATCHLEVEL}"
if command -v shellcheck >/dev/null 2>&1; then shellcheck --version | head -2; else echo "shellcheck: not installed"; fi
if (( BASH_VERSINFO[0] >= 5 && BASH_VERSINFO[1] >= 3 )); then echo "RUNTIME=5.3+: all probes available"; else echo "RUNTIME<5.3: 5.3-only probes self-skip"; fi


## GLOBSORT -- natural sort for pathname expansion

`GLOBSORT` is a `shopt` option introduced in Bash 5.3 that changes the sort order of pathname expansion. The option name is unrecognized on Bash < 5.3 -- `shopt -s GLOBSORT` prints `bash: GLOBSORT: invalid shell option name` and exits non-zero.


In [ ]:
# GLOBSORT demonstration
# GLOBSORT: sort order before and after enabling (Bash 5.3+)
tmpdir=$(mktemp -d)
touch "$tmpdir"/v1 "$tmpdir"/v2 "$tmpdir"/v10 "$tmpdir"/v20

echo "=== Default (lexicographic) ==="
(cd "$tmpdir" && printf "%s\n" v*)

if (( BASH_VERSINFO[0] >= 5 && BASH_VERSINFO[1] >= 3 )); then
  echo "=== With GLOBSORT ==="
  (cd "$tmpdir" && shopt -s GLOBSORT && printf "%s\n" v*)
else
  echo "SKIP: GLOBSORT is 5.3+ only"
fi

rm -rf "$tmpdir"


## inherit_errexit -- propagating errexit into subshells

Without `inherit_errexit`, a command substitution `$(cmd)` runs in a subshell that does **not** inherit the `set -e` flag -- a failing `cmd` inside `$(...)` does not abort the parent script. `shopt -s inherit_errexit` makes subshells and command substitutions inherit `errexit`, so a failure inside `$(cmd)` surfaces as a non-zero exit. The option should be guarded with a `BASH_VERSINFO` check for cross-version safety.


In [ ]:
# inherit_errexit demonstration
# inherit_errexit: set -e propagation into command substitutions
set -e

# Without inherit_errexit (default):
#   set -e is NOT active inside the subshell created by $( ).
#   false fails but the subshell continues to the next command.
output=$(false; echo "subshell continued past false")
echo "output: $output"
echo "Without inherit_errexit: subshell continued (errexit gap exists)"

# With inherit_errexit enabled, subshells inherit set -e:
shopt -s inherit_errexit 2>/dev/null && echo "inherit_errexit: ON" || echo "inherit_errexit: not available"

# Restore for subsequent cells
set +e


## ${| command; } -- forkless command substitution and pipefail

Bash 5.3 introduces `${ command; }` and `${|command;}` -- two forms of command substitution that execute **in the current shell context without forking a child process or using a pipe**. The first captures output into the variable; the second places the result in `REPLY`. Because no pipe is involved, `pipefail` and `PIPESTATUS` concerns are moot -- the inner command's exit code is directly available as `$?` after the substitution returns.


In [ ]:
# forkless command substitution demonstration
# ${ command; } -- forkless command substitution (Bash 5.3+)
# Captures output without forking a child process or using a pipe.
echo "Bash version: ${BASH_VERSION}"
if (( BASH_VERSINFO[0] >= 5 && BASH_VERSINFO[1] >= 3 )); then
  # ${ command; } runs in the current shell context, no fork, no pipe
  captured=${ printf "no fork, no pipe"; }
  echo "captured: $captured  (exit was: $?)"

  # ${|command;} runs in current shell; the command sets $REPLY.
  # Unlike $(cmd | cmd2) this needs no pipe -- so pipefail is irrelevant.
  ${| REPLY="value set in REPLY"; }
  echo "REPLY: $REPLY"
else
  echo "SKIP: forkless command substitution needs Bash 5.3+"
fi


### Why this matters for `pipefail`

The traditional form `output=$(cmd1 | cmd2)` involves a pipe -- the exit code of `cmd1` is masked unless `set -o pipefail` is enabled and inspected via `PIPESTATUS`. The 5.3 `${| command;}` form captures output **without a pipe**, so the inner command's exit status is directly inspectable as `$?` after the substitution. For single-command captures that only existed to avoid `pipefail` complexity, `${ command;}` is a lighter-weight replacement.


## test builtin -- parenthesized subexpression parsing

Bash 5.3 changed how the `test` builtin (and `[`) parses **parenthesized subexpressions** when an expression contains more than four arguments. On Bash 5.2 the parser reads left-to-right and may misinterpret the grouping; on 5.3 the parser **scans forward for the closing paren** and routes the subexpression to `posixtest()` if four or fewer arguments sit between the parens. The behavior is coreutils-compatible but depends on the active shell compatibility level, and the Bash NEWS explicitly notes the expression *remains ambiguous*. Complex expressions that rely on specific parsing should be re-tested on both runtimes.


In [ ]:
# test builtin parsing demonstration
# test/[ builtin: >4-arg parenthesized subexpression parsing changed in 5.3
echo "Bash version: ${BASH_VERSION}"
a=5; b=7; c=3; d=9

# Simple expressions behave identically on 5.2 and 5.3:
[ "$a" -gt 0 ] && [ "$b" -lt 10 ] && echo "simple [ ] chain: ok"
[[ $a -gt 0 && $b -lt 10 ]] && echo "[[ ]] compound: ok"

# Complex expression with parenthesized subexpression (>4 args):
#   [ ( $a -gt 0 -a $b -lt 10 ) -o ( $c -lt 5 -a $d -gt 0 ) ]
# In 5.3 the parser scans forward for the closing paren (coreutils-compatible).
# In 5.2 the same expression may parse left-to-right differently.
# The expression "remains ambiguous" per Bash NEWS -- run on both runtimes
# and inspect $? to determine which branch was taken.
if (( BASH_VERSINFO[0] >= 5 && BASH_VERSINFO[1] >= 3 )); then
  echo "On 5.3+: re-run complex [ ] expressions and compare $? against 5.2 baseline"
else
  echo "SKIP: test parsing change is 5.3-specific (5.2 baseline here)"
fi


## Verify

1. Run `bash --version` and confirm the runtime is Bash 5.3 patchlevel 20 (or the latest available 5.3 patch).
2. Run every code cell; on a 5.3 runtime all probes execute, on 5.2 the 5.3-only cells print `SKIP`.
3. On a 5.2 runtime, save each cell's output; re-run on 5.3 and diff -- any reorder in the `GLOBSORT` cell or difference in `$?` after the `test` cell is a migration finding.
4. Lint each code cell with `bash -n` and `shellcheck -s bash`; the forkless substitution syntax `${| command;}` triggers a warning on ShellCheck < 0.11.0 if the dialect is not set to a 5.3-aware version.


## Common errors

- **`bash: GLOBSORT: invalid shell option name`** -- Running on Bash < 5.3. Guard the `shopt` call with a `BASH_VERSINFO` check before enabling the option.
- **`shopt: inherit_errexit: invalid shell option name`** -- Older Bash does not recognize the option. Guard with a version check; the option should be enabled conditionally, not unconditionally.
- **`bad substitution`** on `${| command; }` -- The forkless form is a syntax error on Bash < 5.3. Guard with the version check in cell 0.
- **`too many arguments`** or unexpected exit code from a complex `[ ]` expression -- On 5.2 the parser may misread parenthesized subexpressions with >4 args. Test on the target runtime before shipping.


## References

- Bash 5.3 GA release and GLOBSORT: https://www.phoronix.com/news/GNU-Bash-5.3
- Patch level 20 and upstream patch history: https://cgit.git.savannah.gnu.org/cgit/bash.git/log/?h=master
- Forkless command substitution (`${ command; }` / `${|command;}`) and CHANGES notes: https://tiswww.case.edu/php/chet/bash/CHANGES
- Bash 5.3 announcement (inherit_errexit noted as shipped in 5.3): https://lists.gnu.org/archive/html/bash-announce/2025-07/msg00000.html
- Bash 5.3 migration gotchas (`test` builtin parsing >4 args): https://lists.gnu.org/archive/html/bash-announce/2025-07/msg00005.html
- Full Bash 5.3 NEWS (feature list, incompatible changes, posixtest): https://cgit.git.savannah.gnu.org/cgit/bash.git/plain/NEWS
